# Assignment 1: Attention Mechanisms from Scratch

## Overview

You will implement the core attention mechanisms that power every modern Transformer -- from scratch,
using only PyTorch tensor operations. No `nn.MultiheadAttention`, no `nn.TransformerEncoder`.
Every matrix multiplication, every softmax, every mask must be your code.

When you finish, you should be able to implement attention on a whiteboard from memory.

### Learning Objectives
- Implement scaled dot-product attention from raw tensor operations
- Build multi-head attention with proper head splitting and concatenation
- Train attention on a copy task and visualize learned attention patterns
- Implement causal masking for autoregressive generation
- Verify your implementation against PyTorch's `nn.MultiheadAttention`

### Key Equation

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$

where $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$

**Estimated time:** 8-12 hours

---
## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import time
import sys
sys.path.insert(0, '../../')

from shared_utils.common import set_seed, get_device

set_seed(42)
device = get_device()
print(f"Using device: {device}")

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

### Helper Functions

In [ ]:
def plot_attention_weights(attention_weights, x_labels=None, y_labels=None, title="Attention Weights"):
    """Plot attention weights as a heatmap.
    
    Args:
        attention_weights: Tensor of shape (seq_len_q, seq_len_k) or (num_heads, seq_len_q, seq_len_k)
        x_labels: Labels for key positions (x-axis)
        y_labels: Labels for query positions (y-axis)
        title: Plot title
    """
    if attention_weights.dim() == 3:
        num_heads = attention_weights.shape[0]
        fig, axes = plt.subplots(1, num_heads, figsize=(4 * num_heads, 4))
        if num_heads == 1:
            axes = [axes]
        for i, ax in enumerate(axes):
            im = ax.imshow(attention_weights[i].detach().cpu().numpy(), cmap='viridis', vmin=0, vmax=1)
            ax.set_title(f'Head {i}')
            if x_labels is not None:
                ax.set_xticks(range(len(x_labels)))
                ax.set_xticklabels(x_labels, rotation=45)
            if y_labels is not None:
                ax.set_yticks(range(len(y_labels)))
                ax.set_yticklabels(y_labels)
            ax.set_xlabel('Key')
            ax.set_ylabel('Query')
        plt.colorbar(im, ax=axes, shrink=0.8)
    else:
        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(attention_weights.detach().cpu().numpy(), cmap='viridis', vmin=0, vmax=1)
        if x_labels is not None:
            ax.set_xticks(range(len(x_labels)))
            ax.set_xticklabels(x_labels, rotation=45)
        if y_labels is not None:
            ax.set_yticks(range(len(y_labels)))
            ax.set_yticklabels(y_labels)
        ax.set_xlabel('Key')
        ax.set_ylabel('Query')
        plt.colorbar(im)
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def generate_copy_data(batch_size: int, seq_len: int, vocab_size: int = 10):
    """Generate data for the copy task.
    
    Args:
        batch_size: Number of sequences
        seq_len: Length of each sequence
        vocab_size: Number of distinct tokens (excluding special tokens)
        
    Returns:
        source: Random integer sequences, shape (batch_size, seq_len)
        target: Same sequences (model must learn to copy), shape (batch_size, seq_len)
    """
    source = torch.randint(0, vocab_size, (batch_size, seq_len))
    target = source.clone()
    return source, target


print("Helper functions loaded.")

---
## Part 1: Scaled Dot-Product Attention

Implement `scaled_dot_product_attention(Q, K, V, mask=None)` using only:
- `torch.matmul` (or `@`)
- `torch.softmax` (or `F.softmax`)
- Basic arithmetic (`/`, `**`)
- `torch.masked_fill` for masking

The mask convention: `True`/`1` at positions to **mask** (set to $-\infty$ before softmax).

In [ ]:
def scaled_dot_product_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    mask: torch.Tensor = None
) -> tuple:
    """Compute scaled dot-product attention.
    
    Args:
        Q: Query tensor, shape (..., seq_len_q, d_k)
        K: Key tensor, shape (..., seq_len_k, d_k)
        V: Value tensor, shape (..., seq_len_k, d_v)
        mask: Optional boolean mask, shape broadcastable to (..., seq_len_q, seq_len_k)
              True at positions to MASK (set to -inf before softmax)
              
    Returns:
        output: Weighted sum of values, shape (..., seq_len_q, d_v)
        attention_weights: Softmax weights, shape (..., seq_len_q, seq_len_k)
    """
    # YOUR CODE HERE
    # Steps:
    # 1. Compute d_k from Q's last dimension
    # 2. Compute attention scores: Q @ K^T / sqrt(d_k)
    # 3. Apply mask if provided (set masked positions to -inf)
    # 4. Apply softmax along the last dimension
    # 5. Compute output: attention_weights @ V
    raise NotImplementedError("Implement scaled_dot_product_attention")

### Test: Scaled Dot-Product Attention

In [ ]:
def test_scaled_dot_product_attention():
    """Run all verification tests for scaled dot-product attention."""
    set_seed(42)
    
    # Test 1: Shape test
    Q = torch.randn(2, 5, 64)
    K = torch.randn(2, 10, 64)
    V = torch.randn(2, 10, 32)
    output, weights = scaled_dot_product_attention(Q, K, V)
    assert output.shape == (2, 5, 32), f"Expected output shape (2, 5, 32), got {output.shape}"
    assert weights.shape == (2, 5, 10), f"Expected weights shape (2, 5, 10), got {weights.shape}"
    print("PASSED: Shape test")
    
    # Test 2: Row sum test -- each row of attention weights should sum to 1.0
    row_sums = weights.sum(dim=-1)
    assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-6), \
        f"Attention weight rows do not sum to 1.0! Max deviation: {(row_sums - 1.0).abs().max():.2e}"
    print("PASSED: Row sum test (all rows sum to 1.0)")
    
    # Test 3: Identity test -- if Q = K, diagonal should have highest values
    X = torch.randn(1, 5, 64)
    _, weights_self = scaled_dot_product_attention(X, X, X)
    diag_values = weights_self[0].diag()
    max_per_row = weights_self[0].max(dim=-1).values
    assert torch.allclose(diag_values, max_per_row, atol=1e-5), \
        "When Q=K, diagonal should have highest attention weights"
    print("PASSED: Identity test (Q=K -> diagonal dominance)")
    
    # Test 4: 4D input (for multi-head usage)
    Q4d = torch.randn(2, 4, 5, 64)
    K4d = torch.randn(2, 4, 10, 64)
    V4d = torch.randn(2, 4, 10, 32)
    output4d, weights4d = scaled_dot_product_attention(Q4d, K4d, V4d)
    assert output4d.shape == (2, 4, 5, 32), f"Expected (2,4,5,32), got {output4d.shape}"
    print("PASSED: 4D input test (batch + heads)")
    
    print("\nAll scaled dot-product attention tests passed!")

test_scaled_dot_product_attention()

---
## Part 2: Multi-Head Attention

Implement a complete `MultiHeadAttention` class from scratch.

Steps inside `forward`:
1. Project Q, K, V using linear layers: $(\text{batch}, \text{seq}, d_{\text{model}}) \to (\text{batch}, \text{seq}, d_{\text{model}})$
2. Reshape to $(\text{batch}, \text{num\_heads}, \text{seq}, d_k)$ where $d_k = d_{\text{model}} / \text{num\_heads}$
3. Apply `scaled_dot_product_attention` per head (in parallel via batched matmul)
4. Concatenate heads: $(\text{batch}, \text{seq}, \text{num\_heads} \times d_k) = (\text{batch}, \text{seq}, d_{\text{model}})$
5. Apply output projection $W^O$

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention implemented from scratch."""
    
    def __init__(self, d_model: int, num_heads: int):
        """Initialize Multi-Head Attention.
        
        Args:
            d_model: Total model dimension (must be divisible by num_heads)
            num_heads: Number of attention heads
        """
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # YOUR CODE HERE
        # Initialize W_Q, W_K, W_V, W_O as nn.Linear(d_model, d_model, bias=False)
        raise NotImplementedError("Initialize MultiHeadAttention projections")
    
    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: torch.Tensor = None
    ) -> tuple:
        """Forward pass of multi-head attention.
        
        Args:
            query: (batch, seq_len_q, d_model)
            key:   (batch, seq_len_k, d_model)
            value: (batch, seq_len_k, d_model)
            mask:  Optional, broadcastable to (batch, num_heads, seq_len_q, seq_len_k)
            
        Returns:
            output: (batch, seq_len_q, d_model)
            attention_weights: (batch, num_heads, seq_len_q, seq_len_k)
        """
        # YOUR CODE HERE
        # 1. Project Q, K, V
        # 2. Reshape: (batch, seq, d_model) -> (batch, seq, num_heads, d_k) -> (batch, num_heads, seq, d_k)
        # 3. Apply scaled_dot_product_attention
        # 4. Reshape back: (batch, num_heads, seq, d_k) -> (batch, seq, d_model)
        #    Remember: call .contiguous() before .view() after transpose
        # 5. Apply output projection W_O
        raise NotImplementedError("Implement MultiHeadAttention.forward")

### Test: Multi-Head Attention

In [ ]:
def test_multi_head_attention():
    """Verify MultiHeadAttention shapes and parameter counts."""
    set_seed(42)
    d_model, num_heads = 512, 8
    mha = MultiHeadAttention(d_model, num_heads)
    
    # Test 1: Shape test
    x = torch.randn(4, 20, d_model)
    output, weights = mha(x, x, x)
    assert output.shape == (4, 20, d_model), f"Expected (4, 20, {d_model}), got {output.shape}"
    assert weights.shape == (4, num_heads, 20, 20), f"Expected (4, {num_heads}, 20, 20), got {weights.shape}"
    print(f"PASSED: Shape test -- output {output.shape}, weights {weights.shape}")
    
    # Test 2: Parameter count (4 * d_model^2 for W_Q, W_K, W_V, W_O without biases)
    n_params = sum(p.numel() for p in mha.parameters())
    expected = 4 * d_model * d_model
    assert n_params == expected, f"Expected {expected} params, got {n_params}"
    print(f"PASSED: Parameter count = {n_params:,} (expected {expected:,})")
    
    # Test 3: Self-attention test
    output_self, _ = mha(x, x, x)
    assert output_self.shape == (4, 20, d_model)
    print("PASSED: Self-attention test")
    
    # Test 4: Cross-attention test
    q = torch.randn(4, 10, d_model)
    kv = torch.randn(4, 20, d_model)
    output_cross, weights_cross = mha(q, kv, kv)
    assert output_cross.shape == (4, 10, d_model), \
        f"Cross-attention: expected (4, 10, {d_model}), got {output_cross.shape}"
    assert weights_cross.shape == (4, num_heads, 10, 20), \
        f"Cross-attention weights: expected (4, {num_heads}, 10, 20), got {weights_cross.shape}"
    print("PASSED: Cross-attention test (query length != key length)")
    
    print("\nAll multi-head attention tests passed!")

test_multi_head_attention()

---
## Part 3: The Copy Task

Train your multi-head attention on a simple copy task to verify that attention can learn to
"attend to the right positions."

- Input: `[3, 7, 1, 5, 9]` (random sequence)
- Target: `[3, 7, 1, 5, 9]` (same sequence -- the model must copy it)

After training, the attention weight matrix should be approximately the identity matrix.

In [ ]:
class CopyModel(nn.Module):
    """Simple model for the copy task: Embedding -> MHA (cross-attention) -> Linear."""
    
    def __init__(self, vocab_size: int, d_model: int, num_heads: int):
        super().__init__()
        # YOUR CODE HERE
        # 1. Source embedding: nn.Embedding(vocab_size, d_model)
        # 2. Target embedding: nn.Embedding(vocab_size, d_model) 
        # 3. MultiHeadAttention(d_model, num_heads)
        # 4. Output linear: nn.Linear(d_model, vocab_size)
        raise NotImplementedError("Implement CopyModel")
    
    def forward(self, source, target):
        """Forward pass.
        
        Args:
            source: Source sequence, (batch, seq_len)
            target: Target sequence, (batch, seq_len) -- used as query
            
        Returns:
            logits: (batch, seq_len, vocab_size)
            attention_weights: (batch, num_heads, seq_len, seq_len)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement CopyModel.forward")

In [ ]:
# Train the copy model
set_seed(42)
vocab_size = 10
d_model = 64
num_heads = 4
seq_len = 8
n_epochs = 200
batch_size = 64

copy_model = CopyModel(vocab_size, d_model, num_heads)
optimizer = torch.optim.Adam(copy_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

losses = []
for epoch in range(n_epochs):
    source, target = generate_copy_data(batch_size, seq_len, vocab_size)
    
    logits, attn_weights = copy_model(source, target)
    loss = criterion(logits.view(-1, vocab_size), target.view(-1))
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.4f}")

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Copy Task Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Visualize attention weights for test sequences
copy_model.eval()
with torch.no_grad():
    test_source, test_target = generate_copy_data(3, seq_len, vocab_size)
    _, test_attn = copy_model(test_source, test_target)
    
    for i in range(3):
        tokens = [str(t.item()) for t in test_source[i]]
        plot_attention_weights(
            test_attn[i],  # (num_heads, seq_len, seq_len)
            x_labels=tokens,
            y_labels=tokens,
            title=f"Copy Task - Sequence {i+1}: {tokens}"
        )

---
## Part 4: Attention Weight Visualization

Create more interesting test cases to build intuition about what attention learns.

1. Sequences with repeated tokens (e.g., `[A, B, C, A, B]`)
2. A "reverse" task: input `[3, 7, 1, 5, 9]`, target `[9, 5, 1, 7, 3]`

In [ ]:
# Test with repeated tokens
repeated_source = torch.tensor([[1, 2, 3, 1, 2]])
repeated_target = repeated_source.clone()

with torch.no_grad():
    _, attn_repeated = copy_model(repeated_source, repeated_target)

tokens = ['1', '2', '3', '1', '2']
plot_attention_weights(
    attn_repeated[0],
    x_labels=tokens,
    y_labels=tokens,
    title="Copy Task with Repeated Tokens [1, 2, 3, 1, 2]"
)

In [ ]:
# Train a reverse task model
# YOUR CODE HERE
# 1. Modify generate_copy_data to return reversed targets
# 2. Train a new CopyModel on the reverse task
# 3. Visualize attention -- should show anti-diagonal pattern

### Attention Visualization Analysis

For each scenario, analyze:
- What has each attention head learned?
- How does the model handle repeated tokens?

**Your analysis here:**

YOUR ANSWER HERE

---
## Part 5: Causal (Masked) Attention

Implement causal attention for autoregressive models. In causal attention, each position can
only attend to itself and previous positions -- never to future positions.

In [ ]:
def create_causal_mask(seq_len: int) -> torch.Tensor:
    """Create an upper-triangular causal mask.
    
    Args:
        seq_len: Sequence length
        
    Returns:
        mask: Boolean tensor of shape (seq_len, seq_len)
              True at positions that should be MASKED (future positions)
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement create_causal_mask")

### Test: Causal Masking

In [ ]:
def test_causal_masking():
    """Verify causal masking produces correct attention patterns."""
    set_seed(42)
    seq_len = 5
    d_k = 16
    
    Q = torch.randn(1, seq_len, d_k)
    K = torch.randn(1, seq_len, d_k)
    V = torch.randn(1, seq_len, d_k)
    
    mask = create_causal_mask(seq_len)
    _, weights = scaled_dot_product_attention(Q, K, V, mask=mask)
    weights = weights[0]  # Remove batch dim
    
    print("Causal attention weights:")
    print(weights.detach().numpy().round(3))
    
    # Verify: all weights above diagonal should be 0
    upper_triangle = torch.triu(weights, diagonal=1)
    assert torch.allclose(upper_triangle, torch.zeros_like(upper_triangle), atol=1e-6), \
        "Weights above diagonal should be 0!"
    print("\nPASSED: No attention to future positions")
    
    # Verify: rows sum to 1
    row_sums = weights.sum(dim=-1)
    assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-6), \
        "Rows should sum to 1.0!"
    print("PASSED: Rows sum to 1.0")
    
    # Verify: token 0 attends only to itself
    assert torch.allclose(weights[0, 0], torch.tensor(1.0), atol=1e-6), \
        "Token 0 should attend only to itself with weight 1.0"
    print("PASSED: Token 0 attends only to itself")
    
    # Visualize
    plot_attention_weights(weights, title="Causal Attention Mask Pattern")
    print("\nAll causal masking tests passed!")

test_causal_masking()

### Autoregressive Generation

In [ ]:
def generate_autoregressive(model, start_token: int, max_len: int, vocab_size: int):
    """Generate a sequence token by token using causal attention.
    
    Args:
        model: Trained model with causal masking support
        start_token: Initial token index
        max_len: Maximum sequence length to generate
        vocab_size: Size of vocabulary
        
    Returns:
        tokens: List of generated token indices
    """
    # YOUR CODE HERE
    # 1. Start with [start_token]
    # 2. At each step, run the model on the current sequence with causal mask
    # 3. Take argmax (or sample) from the last position's logits
    # 4. Append to tokens list
    raise NotImplementedError("Implement generate_autoregressive")

---
## Part 6: Comparison with PyTorch's nn.MultiheadAttention

Verify that your implementation produces the same output as PyTorch's built-in module.

**Note:** PyTorch packs Q, K, V projections into a single `in_proj_weight` matrix of shape
`(3 * d_model, d_model)`, structured as `[W_Q; W_K; W_V]` stacked vertically.

In [ ]:
def compare_with_pytorch(d_model: int = 64, num_heads: int = 4, n_tests: int = 5):
    """Compare your MultiHeadAttention with PyTorch's nn.MultiheadAttention."""
    set_seed(42)
    
    # Create both modules
    your_mha = MultiHeadAttention(d_model, num_heads)
    pytorch_mha = nn.MultiheadAttention(d_model, num_heads, bias=False, batch_first=True)
    
    # Copy weights from your module to PyTorch's module
    # YOUR CODE HERE
    # PyTorch's in_proj_weight = [W_Q; W_K; W_V] of shape (3*d_model, d_model)
    # PyTorch's out_proj.weight = W_O of shape (d_model, d_model)
    with torch.no_grad():
        # Copy your W_Q, W_K, W_V into pytorch_mha.in_proj_weight
        # Copy your W_O into pytorch_mha.out_proj.weight
        pass  # YOUR CODE HERE
    
    # Run comparison tests
    for i in range(n_tests):
        x = torch.randn(4, 10, d_model)
        
        your_output, _ = your_mha(x, x, x)
        pytorch_output, _ = pytorch_mha(x, x, x)
        
        max_diff = (your_output - pytorch_output).abs().max().item()
        assert torch.allclose(your_output, pytorch_output, atol=1e-5), \
            f"Test {i+1}: outputs differ! Max diff: {max_diff:.2e}"
        print(f"Test {i+1}: PASSED (max diff: {max_diff:.2e})")
    
    print("\nAll PyTorch comparison tests passed!")

compare_with_pytorch()

---
## Part 7: Speed Benchmark

Measure the computational cost of your attention implementation vs PyTorch's optimized version.

In [ ]:
def benchmark_attention(seq_lengths, d_model=256, num_heads=8, batch_size=32, n_runs=100):
    """Benchmark your attention vs PyTorch's at different sequence lengths."""
    your_times = []
    pytorch_times = []
    
    for seq_len in seq_lengths:
        your_mha = MultiHeadAttention(d_model, num_heads)
        pytorch_mha = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        
        x = torch.randn(batch_size, seq_len, d_model)
        
        # Warmup
        with torch.no_grad():
            your_mha(x, x, x)
            pytorch_mha(x, x, x)
        
        # Your implementation
        start = time.time()
        for _ in range(n_runs):
            with torch.no_grad():
                your_mha(x, x, x)
        your_time = (time.time() - start) / n_runs
        your_times.append(your_time * 1000)  # Convert to ms
        
        # PyTorch
        start = time.time()
        for _ in range(n_runs):
            with torch.no_grad():
                pytorch_mha(x, x, x)
        pytorch_time = (time.time() - start) / n_runs
        pytorch_times.append(pytorch_time * 1000)
        
        speedup = your_time / pytorch_time
        print(f"Seq len {seq_len:4d}: Yours {your_time*1000:.2f} ms, PyTorch {pytorch_time*1000:.2f} ms, Speedup: {speedup:.1f}x")
    
    return your_times, pytorch_times

seq_lengths = [64, 128, 256, 512, 1024]
your_times, pytorch_times = benchmark_attention(seq_lengths)

In [ ]:
# Plot benchmark results
plt.figure(figsize=(10, 6))
plt.plot(seq_lengths, your_times, 'o-', label='Your Implementation', linewidth=2)
plt.plot(seq_lengths, pytorch_times, 's-', label='PyTorch nn.MultiheadAttention', linewidth=2)
plt.xlabel('Sequence Length')
plt.ylabel('Forward Pass Time (ms)')
plt.title('Attention Forward Pass: Your Implementation vs PyTorch')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Benchmark Analysis

Explain WHY PyTorch's implementation is faster.

**Your analysis here:**

YOUR ANSWER HERE

---
## Stretch Goals (Optional)

1. **Conceptual Flash Attention**: Implement tiling and online softmax in pure PyTorch.
2. **Additive (Bahdanau) Attention**: Compare to dot-product attention on the copy task.
3. **Linear Attention**: O(n) attention with kernel approximation.
4. **Relative Position Bias**: Add learnable relative position bias (as in T5 or ALiBi).

In [ ]:
# Stretch goal workspace

# YOUR STRETCH GOAL CODE HERE